In [1]:
import pandas as pd
import geopandas as gpd
from shapely.geometry import shape
from shapely import wkt
from shapely.wkt import loads


### Load files

In [19]:
pedestrian_space = pd.read_csv("C:/Users/isamu/OneDrive/THESIS/DATA/test_big_poly_ped.csv")
pedestrian_space["geometry"] = pedestrian_space["geometry"].apply(lambda x: loads(x))
pedestrian_space = gpd.GeoDataFrame(pedestrian_space, geometry = "geometry", crs = "EPSG:28992")
pedestrian_space

,Unnamed: 0,geometry
0,0,"MULTIPOLYGON (((110478.605 483088.155, 110481...."


In [2]:
erf = gpd.read_file("C:/Users/isamu/OneDrive/THESIS/DATA/Amsterdam/erf_space.geojson")

In [3]:
private = gpd.read_file("C:/Users/isamu/OneDrive/THESIS/DATA/Amsterdam/private_space.gpkg")

In [4]:
cores_ac = pd.read_csv("C:/Users/isamu/OneDrive/THESIS/DATA/TravelTime/cores_traveltime.csv", index_col = 0)

In [4]:
hotels_ac = pd.read_csv("C:/Users/isamu/OneDrive/THESIS/DATA/TravelTime/hotels_traveltime.csv", index_col = 0)

In [5]:
pt15 = gpd.read_file("C:/Users/isamu/OneDrive/THESIS/DATA/TravelTime/merged_isochrones_pt15.geojson")
pt15 = pt15.rename(columns = {"geometry":"public_transport_15min"})[["place_id", "public_transport_15min"]]

In [7]:
pt15_cores = gpd.read_file("C:/Users/isamu/OneDrive/THESIS/DATA/TravelTime/cores_merged_isochrones_pt15.geojson")
pt15_cores = pt15_cores.rename(columns = {"geometry":"public_transport_15min"})[["place", "public_transport_15min"]]

In [11]:
rudifun_bruto = gpd.read_file("C:/Users/isamu/OneDrive/THESIS/DATA/Amsterdam/RUDIFUN/Bruto_bouwblok_frame.shp")

In [47]:
rudifun_netto = gpd.read_file("C:/Users/isamu/OneDrive/THESIS/DATA/Amsterdam/RUDIFUN/Netto_bouwblok_frame.shp")

### Cores

In [28]:
cores_ac = cores_ac.merge(pt15_cores, on = "place", how = "inner")
cores_ac = cores_ac[["place", "walking_5min", "walking_15min", "walking_25min", 
                       "cycling_5min", "cycling_15min", "public_transport_15min"]]

In [33]:
df = cores_ac.melt(
    id_vars='place', 
    value_vars=cores_ac.columns[1:7],
    var_name='geometry_type', 
    value_name='multipolygon')

In [35]:
#df["multipolygon"] = df["multipolygon"].apply(lambda x: loads(x))

df["multipolygon"] = df["multipolygon"].apply(
    lambda x: loads(x) if isinstance(x, str) else x
)
multi = gpd.GeoDataFrame(df, geometry = "multipolygon", crs = "EPSG:4326")
multi = multi.to_crs("EPSG:28992")

In [36]:
gdf_cores_erf = gpd.overlay(erf, multi, how = 'intersection')

In [46]:
gdf_cores_erf.to_csv("C:/Users/isamu/OneDrive/THESIS/DATA/TravelTime/Spaces/cores_erf.csv")

In [47]:
gdf_cores_private = gpd.overlay(private, multi, how = 'intersection')

In [49]:
gdf_cores_private.to_csv("C:/Users/isamu/OneDrive/THESIS/DATA/TravelTime/Spaces/cores_private.csv")

In [29]:
#gdf_pedestrian = gpd.overlay(pedestrian_space, multi, how = 'intersection')

GEOSException: bad allocation

### Hotels

#### erf

In [8]:
hotels_ac = hotels_ac.merge(pt15, on = "place_id", how = "inner")
hotels_ac = hotels_ac[["place_id", "name", "walking_5min", "walking_15min", "walking_25min", 
                       "cycling_5min", "cycling_15min", "public_transport_15min"]]

In [9]:
df = hotels_ac.melt(
    id_vars=['place_id', 'name'], 
    value_vars=hotels_ac.columns[2:8],
    var_name='geometry_type', 
    value_name='multipolygon')

In [10]:
df["multipolygon"] = df["multipolygon"].apply(
    lambda x: loads(x) if isinstance(x, str) else x
)
multi = gpd.GeoDataFrame(df, geometry = "multipolygon", crs = "EPSG:4326")
multi = multi.to_crs("EPSG:28992")

In [13]:
joined = gpd.sjoin(erf, multi, how="inner", predicate="intersects")

In [23]:
joined["area"] = joined["geometry"].area

In [27]:
hotels_erf = joined.groupby(["place_id", "name", "geometry_type"])["area"].sum().reset_index()

In [31]:
hotels_erf = hotels_erf.merge(multi, on = ["place_id", "name", "geometry_type"], how = "inner")

In [33]:
hotels_erf.to_csv("C:/Users/isamu/OneDrive/THESIS/DATA/TravelTime/Spaces/hotels_erf.csv")

#### private

In [5]:
hotels_ac = hotels_ac.merge(pt15, on = "place_id", how = "inner")
hotels_ac = hotels_ac[["place_id", "name", "walking_5min", "walking_15min", "walking_25min", 
                       "cycling_5min", "cycling_15min", "public_transport_15min"]]

In [6]:
df = hotels_ac.melt(
    id_vars=['place_id', 'name'], 
    value_vars=hotels_ac.columns[2:8],
    var_name='geometry_type', 
    value_name='multipolygon')

In [7]:
df["multipolygon"] = df["multipolygon"].apply(
    lambda x: loads(x) if isinstance(x, str) else x
)
multi = gpd.GeoDataFrame(df, geometry = "multipolygon", crs = "EPSG:4326")
multi = multi.to_crs("EPSG:28992")

In [9]:
multi.iloc[0:1500,:]

,place_id,name,geometry_type,multipolygon
0,ChIJg8bf-sIJxkcRsvcYZBHfZ14,The Dylan Amsterdam,walking_5min,"MULTIPOLYGON (((120469.393 486772.663, 120469...."
1,ChIJIYdU8sMJxkcRrbxZuGZOlro,The Hoxton Amsterdam,walking_5min,"MULTIPOLYGON (((120622.466 487263.316, 120628...."
2,ChIJozDTD8MJxkcRO_HxQ1B4wF4,Grand Canal Boutique Hotel,walking_5min,"MULTIPOLYGON (((120447.398 486972.134, 120437...."
3,ChIJz-L17MMJxkcRZdrqv-u57P4,Hotel HEGRA by Stanley Collection,walking_5min,"MULTIPOLYGON (((120672.946 487244.162, 120682...."
4,ChIJo8-ANMIJxkcRi1HeWcbBcmI,Ambassade Hotel - Amsterdam,walking_5min,"MULTIPOLYGON (((120722.674 486970.6, 120722.60..."
...,...,...,...,...
1495,ChIJqfK327gJxkcRfvN3QYUBX3A,Hotel 83,cycling_5min,"MULTIPOLYGON (((120858.804 487567.59, 120868.8..."
1496,ChIJ3-qc2rgJxkcRTfGKUgOvAkk,Hotel Torenzicht,cycling_5min,"MULTIPOLYGON (((120938.099 487503.093, 120937...."
1497,ChIJx11JNLgJxkcRQjL4_mmO9ZM,Hotel Luxer,cycling_5min,"MULTIPOLYGON (((120778.409 487635.751, 120803...."
1498,ChIJ-_VFLbgJxkcR6Io_ONVcqX0,Hotel Prins Hendrik,cycling_5min,"MULTIPOLYGON (((121020.23 486993.442, 121010.2..."


In [10]:
gdf_hotels_private = gpd.overlay(private, multi.iloc[0:1500,:], how = 'intersection')

In [11]:
gdf_hotels_private2 = gpd.overlay(private, multi.iloc[1500:,:], how = 'intersection')

In [18]:
hotels_private = pd.concat([gdf_hotels_private, gdf_hotels_private2])

In [21]:
hotels_private["area"] = hotels_private["geometry"].area
hotels_private = hotels_private.groupby(["place_id", "name", "geometry_type"])["area"].sum().reset_index()

In [23]:
hotels_private = hotels_private.merge(multi, on = ["place_id", "name", "geometry_type"], how = "inner")

In [26]:
hotels_private

,place_id,name,geometry_type,area,multipolygon
0,ChIJ-1hUL74JxkcRnLpTBTbfH5o,Eden Hotel Amsterdam,cycling_15min,1.111974e+07,"MULTIPOLYGON (((118568.85 485087.466, 118586.7..."
1,ChIJ-1hUL74JxkcRnLpTBTbfH5o,Eden Hotel Amsterdam,cycling_5min,9.837289e+05,"MULTIPOLYGON (((121488.059 485808.633, 121508...."
2,ChIJ-1hUL74JxkcRnLpTBTbfH5o,Eden Hotel Amsterdam,public_transport_15min,2.495517e+06,"MULTIPOLYGON (((121030.349 484975.141, 121018...."
3,ChIJ-1hUL74JxkcRnLpTBTbfH5o,Eden Hotel Amsterdam,walking_15min,1.302116e+06,"MULTIPOLYGON (((120713.314 486743.952, 120708...."
4,ChIJ-1hUL74JxkcRnLpTBTbfH5o,Eden Hotel Amsterdam,walking_25min,3.598217e+06,"MULTIPOLYGON (((120071.119 486780.097, 120070...."
...,...,...,...,...,...
2503,ChIJzeKdC8AJxkcR01jCuc6AJfw,Hotel Residence Le Coin,cycling_5min,1.195044e+06,"MULTIPOLYGON (((121529.731 485822.677, 121529...."
2504,ChIJzeKdC8AJxkcR01jCuc6AJfw,Hotel Residence Le Coin,public_transport_15min,2.949177e+06,"MULTIPOLYGON (((121055.471 484102.067, 121015...."
2505,ChIJzeKdC8AJxkcR01jCuc6AJfw,Hotel Residence Le Coin,walking_15min,1.520870e+06,"MULTIPOLYGON (((120398.426 486751.149, 120403...."
2506,ChIJzeKdC8AJxkcR01jCuc6AJfw,Hotel Residence Le Coin,walking_25min,3.916645e+06,"MULTIPOLYGON (((119766.753 486616.794, 119776...."


In [25]:
hotels_private.to_csv("C:/Users/isamu/OneDrive/THESIS/DATA/TravelTime/Spaces/hotels_private.csv")

### RUDIFUN

#### Bruto Bouwblok

In [6]:
hotels_ac = hotels_ac.merge(pt15, on = "place_id", how = "inner")
hotels_ac = hotels_ac[["place_id", "name", "walking_5min", "walking_15min", "walking_25min", 
                       "cycling_5min", "cycling_15min", "public_transport_15min"]]

In [7]:
df = hotels_ac.melt(
    id_vars=['place_id', 'name'], 
    value_vars=hotels_ac.columns[2:8],
    var_name='geometry_type', 
    value_name='multipolygon')

In [9]:
df["multipolygon"] = df["multipolygon"].apply(
    lambda x: loads(x) if isinstance(x, str) else x
)
multi = gpd.GeoDataFrame(df, geometry = "multipolygon", crs = "EPSG:4326")
multi = multi.to_crs("EPSG:28992")

In [16]:
gdf_hotels_rudifun = gpd.overlay(rudifun_bruto, multi, how = 'intersection')

In [31]:
gdf_hotels_rudifun_mean = gdf_hotels_rudifun.groupby(["place_id", "name"])[["FSI_24", "GSI_24", "OSR_24", 
                                                                            "L_24", "MXI_24", "SDI_24"]].mean().reset_index()

In [37]:
gdf_hotels_rudifun_mean_tt = gdf_hotels_rudifun.groupby(["place_id", "name", "geometry_type"])[
    ["FSI_24", "GSI_24", "OSR_24", "L_24", "MXI_24", "SDI_24"]].mean().reset_index()

In [40]:
gdf_hotels_rudifun_mean_tt_wide = gdf_hotels_rudifun_mean_tt.pivot_table(
    index=["place_id", "name"],
    columns="geometry_type",
    values=["FSI_24", "GSI_24", "OSR_24", "L_24", "MXI_24", "SDI_24"]
)

# Step 2: flatten MultiIndex columns
gdf_hotels_rudifun_mean_tt_wide.columns = [f"{metric}_{geom}" for metric, geom in gdf_hotels_rudifun_mean_tt_wide.columns]
gdf_hotels_rudifun_mean_tt_wide = gdf_hotels_rudifun_mean_tt_wide.reset_index()

In [43]:
#gdf_hotels_rudifun.to_csv("C:/Users/isamu/OneDrive/THESIS/DATA/TravelTime/Spaces/hotels_bruto_bouwblok.csv")
#gdf_hotels_rudifun_mean.to_csv("C:/Users/isamu/OneDrive/THESIS/DATA/TravelTime/Spaces/hotels_bruto_bouwblok_mean.csv")
#gdf_hotels_rudifun_mean_tt_wide.to_csv("C:/Users/isamu/OneDrive/THESIS/DATA/TravelTime/Spaces/hotels_bruto_bouwblok_mean_tt_wide.csv")

#### Netto Bouwblok

In [ ]:
hotels_ac = hotels_ac.merge(pt15, on = "place_id", how = "inner")
hotels_ac = hotels_ac[["place_id", "name", "walking_5min", "walking_15min", "walking_25min", 
                       "cycling_5min", "cycling_15min", "public_transport_15min"]]

In [45]:
df = hotels_ac.melt(
    id_vars=['place_id', 'name'], 
    value_vars=hotels_ac.columns[2:8],
    var_name='geometry_type', 
    value_name='multipolygon')

In [46]:
df["multipolygon"] = df["multipolygon"].apply(
    lambda x: loads(x) if isinstance(x, str) else x
)
multi = gpd.GeoDataFrame(df, geometry = "multipolygon", crs = "EPSG:4326")
multi = multi.to_crs("EPSG:28992")

In [48]:
gdf_hotels_rudifun = gpd.overlay(rudifun_netto, multi, how = 'intersection')

In [49]:
gdf_hotels_rudifun_mean = gdf_hotels_rudifun.groupby(["place_id", "name"])[["FSI_24", "GSI_24", "OSR_24", 
                                                                            "L_24", "MXI_24", "SDI_24"]].mean().reset_index()

In [50]:
gdf_hotels_rudifun_mean_tt = gdf_hotels_rudifun.groupby(["place_id", "name", "geometry_type"])[
    ["FSI_24", "GSI_24", "OSR_24", "L_24", "MXI_24", "SDI_24"]].mean().reset_index()

In [51]:
gdf_hotels_rudifun_mean_tt_wide = gdf_hotels_rudifun_mean_tt.pivot_table(
    index=["place_id", "name"],
    columns="geometry_type",
    values=["FSI_24", "GSI_24", "OSR_24", "L_24", "MXI_24", "SDI_24"]
)

# Step 2: flatten MultiIndex columns
gdf_hotels_rudifun_mean_tt_wide.columns = [f"{metric}_{geom}" for metric, geom in gdf_hotels_rudifun_mean_tt_wide.columns]
gdf_hotels_rudifun_mean_tt_wide = gdf_hotels_rudifun_mean_tt_wide.reset_index()

In [52]:
gdf_hotels_rudifun.to_csv("C:/Users/isamu/OneDrive/THESIS/DATA/TravelTime/Spaces/hotels_netto_bouwblok.csv")
gdf_hotels_rudifun_mean.to_csv("C:/Users/isamu/OneDrive/THESIS/DATA/TravelTime/Spaces/hotels_netto_bouwblok_mean.csv")
gdf_hotels_rudifun_mean_tt_wide.to_csv("C:/Users/isamu/OneDrive/THESIS/DATA/TravelTime/Spaces/hotels_netto_bouwblok_mean_tt_wide.csv")

### Calculate open space and erf in high capacity areas

In [35]:
hotels_private = pd.read_csv("C:/Users/isamu/OneDrive/THESIS/DATA/TravelTime/Spaces/hotels_private.csv", index_col = 0)
hotels_private = hotels_private.rename(columns = {"area":"area_private"})

# Assuming your dataframe is named 'df'
hotels_private = hotels_private.pivot_table(
    index=['place_id', 'name'],
    columns='geometry_type',
    values='area_private'
).reset_index()

# Flatten the column names
hotels_private.columns.name = None  # Remove columns' name
hotels_private.columns = ['place_id', 'name'] + [f"{col}_area_private" for col in hotels_private.columns[2:]]

In [36]:
hotels_private

,place_id,name,cycling_15min_area_private,cycling_5min_area_private,public_transport_15min_area_private,walking_15min_area_private,walking_25min_area_private,walking_5min_area_private
0,ChIJ-1hUL74JxkcRnLpTBTbfH5o,Eden Hotel Amsterdam,1.111974e+07,9.837289e+05,2.495517e+06,1.302116e+06,3.598217e+06,59794.407710
1,ChIJ-8C0ZsQJxkcR6f0pc6CA3IM,Hotel De Westertoren,1.006002e+07,1.008103e+06,1.397852e+06,1.387666e+06,3.578037e+06,87241.068394
2,ChIJ-8DbTpYJxkcRF6GXegHzFe8,Hotel Hermitage Amsterdam,1.050448e+07,9.074643e+05,1.668253e+06,1.224461e+06,3.418910e+06,46023.901586
3,ChIJ-RAyRD0KxkcRG4koK51opDE,Hotel Novotel Amsterdam City,8.330517e+06,4.560987e+05,1.144943e+06,4.864174e+05,1.826487e+06,33533.421767
4,ChIJ-RS-x8cJxkcRix6HMEVbC0c,Avenue Hotel,1.013871e+07,1.090271e+06,2.553817e+06,1.229657e+06,2.964664e+06,121059.156957
...,...,...,...,...,...,...,...,...
413,ChIJzUkvQZUJxkcR7vLn27sChg0,SWEETS hotel Westerdoksbrug,7.738015e+06,3.454822e+05,1.015818e+06,7.382034e+05,2.273882e+06,40987.339969
414,ChIJzVN3QY_hxUcREIzfWnT4u94,Hotel Artemis Amsterdam,7.254728e+06,7.677193e+05,8.077375e+05,7.511374e+05,1.839191e+06,50980.929683
415,ChIJzVtZ7PkJxkcRFn8elAS0PHg,The Delphi - Amsterdam Townhouse,1.159055e+07,1.145820e+06,1.653182e+06,1.311448e+06,3.593044e+06,100816.350745
416,ChIJzcu9uuUJxkcRsinjE9Hf-L4,Sonder Park House,1.275031e+07,1.062430e+06,2.366779e+06,1.329216e+06,4.083796e+06,80329.966764


In [33]:
hotels_erf = pd.read_csv("C:/Users/isamu/OneDrive/THESIS/DATA/TravelTime/Spaces/hotels_erf.csv", index_col = 0)
hotels_erf = hotels_erf.rename(columns = {"area":"area_erf"})

# Assuming your dataframe is named 'df'
hotels_erf = hotels_erf.pivot_table(
    index=['place_id', 'name'],
    columns='geometry_type',
    values='area_erf'
).reset_index()

# Flatten the column names
hotels_erf.columns.name = None  # Remove columns' name
hotels_erf.columns = ['place_id', 'name'] + [f"{col}_area_erf" for col in hotels_erf.columns[2:]]

In [34]:
hotels_erf

,place_id,name,cycling_15min_area_erf,cycling_5min_area_erf,public_transport_15min_area_erf,walking_15min_area_erf,walking_25min_area_erf,walking_5min_area_erf
0,ChIJ-1hUL74JxkcRnLpTBTbfH5o,Eden Hotel Amsterdam,3.879469e+06,227494.666468,680021.215704,340027.024767,9.566442e+05,13953.642035
1,ChIJ-8C0ZsQJxkcR6f0pc6CA3IM,Hotel De Westertoren,3.541499e+06,203745.595213,283381.763272,282136.703006,9.350009e+05,23599.811639
2,ChIJ-8DbTpYJxkcRF6GXegHzFe8,Hotel Hermitage Amsterdam,3.628995e+06,317228.889479,505406.112921,399622.475593,9.508657e+05,21729.586105
3,ChIJ-RAyRD0KxkcRG4koK51opDE,Hotel Novotel Amsterdam City,4.004236e+06,190340.879052,445428.363171,198352.068498,8.135720e+05,15496.581090
4,ChIJ-RS-x8cJxkcRix6HMEVbC0c,Avenue Hotel,3.461903e+06,162040.793846,587130.629308,199652.349488,6.751507e+05,6192.927487
...,...,...,...,...,...,...,...,...
413,ChIJzUkvQZUJxkcR7vLn27sChg0,SWEETS hotel Westerdoksbrug,2.636213e+06,67854.526193,245911.672924,155905.912716,4.982081e+05,14543.612260
414,ChIJzVN3QY_hxUcREIzfWnT4u94,Hotel Artemis Amsterdam,3.868785e+06,504611.979929,517780.045264,491067.436767,1.121165e+06,82926.181921
415,ChIJzVtZ7PkJxkcRFn8elAS0PHg,The Delphi - Amsterdam Townhouse,4.584572e+06,521521.713977,713432.716348,573221.915519,1.397451e+06,60939.491779
416,ChIJzcu9uuUJxkcRsinjE9Hf-L4,Sonder Park House,4.569593e+06,404395.599040,882656.052909,484886.500498,1.393799e+06,39276.714247


In [37]:
hotels_all = pd.read_csv("C:/Users/isamu/OneDrive/THESIS/DATA/hotels_all_data.csv", index_col = 0)
hotels_all = hotels_all.merge(hotels_private, on = ["place_id", "name"], how = "inner")
hotels_all = hotels_all.merge(hotels_erf, on = ["place_id", "name"], how = "inner")
hotels_all

,place_id,name,5min % high A.C.,15min % high A.C.,25min % high A.C.,Buurt,Wijk,geometry,WKT_LNG_LAT,pressure 5min,...,public_transport_15min_area_private,walking_15min_area_private,walking_25min_area_private,walking_5min_area_private,cycling_15min_area_erf,cycling_5min_area_erf,public_transport_15min_area_erf,walking_15min_area_erf,walking_25min_area_erf,walking_5min_area_erf
0,ChIJ-1hUL74JxkcRnLpTBTbfH5o,Eden Hotel Amsterdam,18.456850,23.195140,26.018651,Rembrandtplein e.o.,Grachtengordel-Zuid,POINT (4.8986899 52.3668376),"POLYGON((4.892818 52.36518,4.895874 52.364754,...",5009.061722,...,2.495517e+06,1.302116e+06,3.598217e+06,59794.407710,3.879469e+06,227494.666468,680021.215704,340027.024767,9.566442e+05,13953.642035
1,ChIJ-8C0ZsQJxkcR6f0pc6CA3IM,Hotel De Westertoren,8.903487,23.484635,25.645533,Felix Meritisbuurt,Grachtengordel-West,POINT (4.8861425999999994 52.3734604),"POLYGON((4.882614 52.368932,4.884411 52.368862...",4822.931316,...,1.397852e+06,1.387666e+06,3.578037e+06,87241.068394,3.541499e+06,203745.595213,283381.763272,282136.703006,9.350009e+05,23599.811639
2,ChIJ-8DbTpYJxkcRF6GXegHzFe8,Hotel Hermitage Amsterdam,15.403297,23.963324,26.182952,Weesperbuurt,Weesperbuurt/Plantage,POINT (4.9036111 52.3644444),"POLYGON((4.901217 52.365779,4.901557 52.365162...",447.224360,...,1.668253e+06,1.224461e+06,3.418910e+06,46023.901586,3.628995e+06,317228.889479,505406.112921,399622.475593,9.508657e+05,21729.586105
3,ChIJ-RAyRD0KxkcRG4koK51opDE,Hotel Novotel Amsterdam City,24.843087,27.637239,27.644549,De Klenckebuurt,Buitenveldert-Oost,POINT (4.8885168 52.3337156),"POLYGON((4.878652 52.335088,4.879167 52.334426...",17.727786,...,1.144943e+06,4.864174e+05,1.826487e+06,33533.421767,4.004236e+06,190340.879052,445428.363171,198352.068498,8.135720e+05,15496.581090
4,ChIJ-RS-x8cJxkcRix6HMEVbC0c,Avenue Hotel,8.965566,22.674889,25.467650,Nieuwendijk-Noord,Burgwallen-Nieuwe Zijde,POINT (4.8946467 52.37665),"POLYGON((4.893582 52.376221,4.8951 52.376206,4...",12037.833635,...,2.553817e+06,1.229657e+06,2.964664e+06,121059.156957,3.461903e+06,162040.793846,587130.629308,199652.349488,6.751507e+05,6192.927487
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
381,ChIJzShofw_ixUcRlkCGMjr3iss,Conscious Hotel Vondelpark,28.162629,29.033888,27.222098,Vondelparkbuurt-West,Overtoomse Sluis,POINT (4.856571199999999 52.3578415),"POLYGON((4.854123 52.356523,4.855085 52.35641,...",707.003344,...,1.789184e+06,1.049075e+06,3.366825e+06,74664.543087,4.636906e+06,333307.171144,732272.755641,451106.113440,1.369786e+06,30137.579240
382,ChIJzUkvQZUJxkcR7vLn27sChg0,SWEETS hotel Westerdoksbrug,13.396704,21.903330,23.892418,Westerdokseiland,Haarlemmerbuurt,POINT (4.8933493 52.383326999999994),"POLYGON((4.890186 52.382395,4.890637 52.382179...",206.150962,...,1.015818e+06,7.382034e+05,2.273882e+06,40987.339969,2.636213e+06,67854.526193,245911.672924,155905.912716,4.982081e+05,14543.612260
383,ChIJzVtZ7PkJxkcRFn8elAS0PHg,The Delphi - Amsterdam Townhouse,21.154826,21.546693,29.000832,Minervabuurt-Noord,Apollobuurt,POINT (4.8769268 52.3504286),"POLYGON((4.866777 52.350394,4.867401 52.348677...",1683.174588,...,1.653182e+06,1.311448e+06,3.593044e+06,100816.350745,4.584572e+06,521521.713977,713432.716348,573221.915519,1.397451e+06,60939.491779
384,ChIJzcu9uuUJxkcRsinjE9Hf-L4,Sonder Park House,13.452959,27.180408,28.292420,P.C. Hooftbuurt,Museumkwartier,POINT (4.8778483 52.360143),"POLYGON((4.877154 52.360101,4.878568 52.358153...",1184.120210,...,2.366779e+06,1.329216e+06,4.083796e+06,80329.966764,4.569593e+06,404395.599040,882656.052909,484886.500498,1.393799e+06,39276.714247


In [39]:
hotels_all.to_csv("C:/Users/isamu/OneDrive/THESIS/DATA/hotels_all_data_erf_and_private.csv")